# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
This dataset is published using the Croissant schema, available via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Publication date: {metadata.datePublished}")
print(f"Dataset citation: {metadata.citeAs}")

## 2. Data Overview
Let's review the record sets, fields, and their `@id` identifiers defined in the dataset. `mlcroissant` exposes metadata about record sets and fields for accessing data elements by their unique `@id`.

In [ ]:
# List record sets and their fields by @id
print("Record Sets (by @id):\n")
for record_set in dataset.record_sets:
    print(f"- Record Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - {field['@id']}")
    print()

Let's view an example record from each record set. You'll see the keys (which are also the field `@id`s) and a sample of values.

In [ ]:
# Show one sample record from each record set
for record_set in dataset.record_sets:
    rset_id = record_set['@id']
    print(f"\nFirst record for record set: {rset_id}")
    records_iter = dataset.records(record_set=rset_id)
    try:
        first_record = next(records_iter)
        print(first_record)
    except StopIteration:
        print("(No data records found)")


## 3. Data Extraction
Now, let's extract data from a chosen record set into a Pandas DataFrame. We reference record sets and fields by their `@id` as recommended by the schema.

In [ ]:
# For this dataset, there is typically a main record set containing protein table data. Let's list their @id:
record_set_ids = [rec_set['@id'] for rec_set in dataset.record_sets]
print("Record set IDs:", record_set_ids)

# Let's use the first record set (most likely the main protein records table)
main_record_set_id = record_set_ids[0]

# Load the complete data for this record set
records = list(dataset.records(record_set=main_record_set_id))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from record set: {main_record_set_id}")
print("Columns (field @id):", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
In this section we'll:
- Filter protein records based on a numeric field (e.g. peptide count or molecular weight field, referenced by its field `@id`).
- Normalize the field.
- Optionally group by a categorical field (e.g. sample type or protein description).

⚠️ _Be sure to adapt field `@id`s below to those listed for this dataset._

In [ ]:
# Replace these with real @id values found above for numeric and group fields
numeric_field_id = None
for c in df.columns:
    if any(w in c.lower() for w in ["peptide", "molecular", "abundance", "weight", "count"]):
        numeric_field_id = c
        break
if not numeric_field_id:
    print("No apparent numeric field found automatically. Set numeric_field_id manually.")
else:
    print(f"Selected numeric field: {numeric_field_id}\n")

# Filtering
threshold = df[numeric_field_id].astype(float).mean() if numeric_field_id else 0
filtered_df = df[df[numeric_field_id].astype(float) > threshold] if numeric_field_id else df
print(f"Filtered to {len(filtered_df)} records where {numeric_field_id} > mean ({threshold:.2f}):")
print(filtered_df.head())

# Normalizing
if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If there's a good groupby candidate, show grouped means
group_field_id = None
candidate_group_fields = [c for c in df.columns if c != numeric_field_id and ('description' in c.lower() or 'sample' in c.lower() or 'type' in c.lower())]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field. We use matplotlib for plotting.

In [ ]:
import matplotlib.pyplot as plt

field = numeric_field_id if numeric_field_id else df.columns[0]

plt.figure(figsize=(8,4))
df[field].astype(float).hist(bins=30, alpha=0.7)
plt.title(f"Distribution of {field}")
plt.xlabel(field)
plt.ylabel("Count")
plt.show()

## 6. Conclusion
In this notebook, we explored a FAIR^2 Croissant-based mass spectrometry dataset. We've shown how to reference any data entity by its `@id`, extract data into Pandas, perform filtering and normalization, and create visualizations to understand protein data properties.

You can now adapt this notebook to perform more detailed analysis, such as studying specific protein features, relating abundances to conditions, or integrating with external biological databases.